# 🚦 Train YOLOv8 — Nhận Diện Biển Báo Giao Thông

**Cuộc thi "Phát triển hệ thống nhận diện giao thông" — Ngày hội CNTT BETU 2026**

---

### 📋 Notebook này gồm các bước:
1. Cài đặt thư viện
2. Tải và chuẩn bị dataset GTSRB
3. Chuyển dataset sang format YOLO
4. Huấn luyện model YOLOv8
5. Đánh giá kết quả
6. Test nhận diện trên ảnh mới
7. Tải model về máy

### ⚠️ Yêu cầu:
- Chọn **Runtime → Change runtime type → GPU (T4)** trước khi chạy
- Chạy **từng ô (cell) từ trên xuống dưới**

---

## 📌 Bước 0: Kiểm tra GPU

In [ ]:
# Kiểm tra GPU đã được bật chưa
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    print(f"✅ GPU đã sẵn sàng: {gpu_name}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("❌ KHÔNG có GPU! Vào Runtime → Change runtime type → GPU (T4)")
    print("   Sau đó chạy lại cell này.")

## 📌 Bước 1: Cài đặt thư viện

In [ ]:
# Cài đặt Ultralytics (YOLOv8)
!pip install -q ultralytics
!pip install -q gdown

print("\n✅ Đã cài đặt xong thư viện!")

## 📌 Bước 2: Tải dataset GTSRB

**GTSRB** (German Traffic Sign Recognition Benchmark):
- 43 loại biển báo giao thông
- ~50.000 ảnh training
- ~12.000 ảnh test

> 💡 **Mẹo cộng điểm:** Sau khi chạy xong notebook này, hãy thêm ảnh biển báo Việt Nam vào dataset để cộng 10% điểm!

In [ ]:
import os
import shutil
import zipfile
import urllib.request
from pathlib import Path

# Tạo thư mục làm việc
WORK_DIR = "/content/traffic_signs"
os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)

print(f"📂 Thư mục làm việc: {WORK_DIR}")
print("📥 Đang tải dataset GTSRB... (có thể mất 2-5 phút)")

# Tải GTSRB Training data
train_url = "https://sid.erda.dk/public/archives/daaeac0d7ce1152aea9b61d9f1e19370/GTSRB_Final_Training_Images.zip"
train_zip = "GTSRB_Training.zip"

if not os.path.exists(train_zip):
    print("   Đang tải training data...")
    urllib.request.urlretrieve(train_url, train_zip)
    print("   ✅ Đã tải xong training data")

# Tải GTSRB Test data
test_url = "https://sid.erda.dk/public/archives/daaeac0d7ce1152aea9b61d9f1e19370/GTSRB_Final_Test_Images.zip"
test_zip = "GTSRB_Test.zip"

if not os.path.exists(test_zip):
    print("   Đang tải test data...")
    urllib.request.urlretrieve(test_url, test_zip)
    print("   ✅ Đã tải xong test data")

# Giải nén
print("📦 Đang giải nén...")
for zf in [train_zip, test_zip]:
    with zipfile.ZipFile(zf, 'r') as z:
        z.extractall(".")

print("✅ Đã tải và giải nén dataset GTSRB thành công!")

## 📌 Bước 3: Chuyển dataset sang format YOLO

YOLOv8 cần dataset theo cấu trúc:
```
dataset/
├── images/
│   ├── train/
│   └── val/
└── labels/
    ├── train/
    └── val/
```

Mỗi file label `.txt` chứa: `class_id x_center y_center width height` (tọa độ chuẩn hóa 0-1)

In [ ]:
import csv
import cv2
import random
from glob import glob

# Tên 43 loại biển báo GTSRB
GTSRB_CLASSES = [
    "Speed limit 20", "Speed limit 30", "Speed limit 50", "Speed limit 60",
    "Speed limit 70", "Speed limit 80", "End speed 80", "Speed limit 100",
    "Speed limit 120", "No passing", "No passing >3.5t", "Priority road",
    "Priority at intersection", "Yield", "Stop", "No vehicles",
    "No vehicles >3.5t", "No entry", "General caution", "Dangerous left",
    "Dangerous right", "Double curve", "Bumpy road", "Slippery road",
    "Road narrows right", "Road work", "Traffic signals", "Pedestrians",
    "Children crossing", "Bicycles crossing", "Beware ice/snow", "Wild animals",
    "End speed+passing", "Turn right ahead", "Turn left ahead", "Ahead only",
    "Go straight or right", "Go straight or left", "Keep right", "Keep left",
    "Roundabout", "End no passing", "End no passing >3.5t"
]

print(f"📊 Tổng số loại biển báo: {len(GTSRB_CLASSES)}")

# Tạo thư mục YOLO
DATASET_DIR = os.path.join(WORK_DIR, "dataset_yolo")
for split in ["train", "val"]:
    os.makedirs(os.path.join(DATASET_DIR, "images", split), exist_ok=True)
    os.makedirs(os.path.join(DATASET_DIR, "labels", split), exist_ok=True)

print("📁 Đã tạo cấu trúc thư mục YOLO")
print("🔄 Đang chuyển đổi dataset... (mất 3-5 phút)")

# Đọc ảnh GTSRB và chuyển sang format YOLO
train_base = os.path.join(WORK_DIR, "GTSRB", "Final_Training", "Images")

all_images = []
count = 0

for class_id in range(43):
    class_dir = os.path.join(train_base, f"{class_id:05d}")
    if not os.path.exists(class_dir):
        continue
    
    # Đọc file CSV annotation
    csv_files = glob(os.path.join(class_dir, "*.csv"))
    if not csv_files:
        # Nếu không có CSV, đọc trực tiếp ảnh
        images = glob(os.path.join(class_dir, "*.ppm")) + glob(os.path.join(class_dir, "*.png")) + glob(os.path.join(class_dir, "*.jpg"))
        for img_path in images:
            img = cv2.imread(img_path)
            if img is None:
                continue
            h, w = img.shape[:2]
            # Toàn ảnh là biển báo → label chiếm gần hết ảnh
            all_images.append((img_path, class_id, 0.5, 0.5, 0.9, 0.9, img))
            count += 1
    else:
        with open(csv_files[0], 'r') as f:
            # Thử đọc với các delimiter phổ biến
            first_line = f.readline()
            f.seek(0)
            delimiter = ';' if ';' in first_line else ','
            reader = csv.reader(f, delimiter=delimiter)
            header = next(reader)  # Skip header
            
            for row in reader:
                try:
                    filename = row[0]
                    img_w, img_h = int(row[1]), int(row[2])
                    x1, y1 = int(row[3]), int(row[4])
                    x2, y2 = int(row[5]), int(row[6])
                    
                    # Chuyển sang YOLO format (chuẩn hóa 0-1)
                    x_center = ((x1 + x2) / 2) / img_w
                    y_center = ((y1 + y2) / 2) / img_h
                    box_w = (x2 - x1) / img_w
                    box_h = (y2 - y1) / img_h
                    
                    img_path = os.path.join(class_dir, filename)
                    if os.path.exists(img_path):
                        img = cv2.imread(img_path)
                        if img is not None:
                            all_images.append((img_path, class_id, x_center, y_center, box_w, box_h, img))
                            count += 1
                except (ValueError, IndexError):
                    continue

print(f"   Đã đọc {count} ảnh từ GTSRB")

# Shuffle và chia train/val (80/20)
random.seed(42)
random.shuffle(all_images)

split_idx = int(len(all_images) * 0.8)
train_data = all_images[:split_idx]
val_data = all_images[split_idx:]

print(f"   Train: {len(train_data)} ảnh | Val: {len(val_data)} ảnh")

# Ghi file ảnh và label
def save_yolo_data(data_list, split_name):
    for i, (img_path, cls_id, xc, yc, bw, bh, img) in enumerate(data_list):
        # Lưu ảnh (chuyển sang JPG)
        img_name = f"{split_name}_{i:06d}.jpg"
        img_save_path = os.path.join(DATASET_DIR, "images", split_name, img_name)
        cv2.imwrite(img_save_path, img)
        
        # Lưu label
        label_name = f"{split_name}_{i:06d}.txt"
        label_save_path = os.path.join(DATASET_DIR, "labels", split_name, label_name)
        with open(label_save_path, 'w') as f:
            f.write(f"{cls_id} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}\n")
        
        if (i + 1) % 5000 == 0:
            print(f"   {split_name}: {i+1}/{len(data_list)}")

print("\n💾 Đang lưu ảnh và label...")
save_yolo_data(train_data, "train")
save_yolo_data(val_data, "val")

print("\n✅ Đã chuyển đổi dataset sang format YOLO thành công!")

In [ ]:
# Tạo file cấu hình dataset cho YOLOv8
yaml_content = f"""# Dataset GTSRB cho YOLOv8
# German Traffic Sign Recognition Benchmark

path: {DATASET_DIR}
train: images/train
val: images/val

# Số lượng class
nc: 43

# Tên các class
names:
"""

for i, name in enumerate(GTSRB_CLASSES):
    yaml_content += f"  {i}: \"{name}\"\n"

yaml_path = os.path.join(WORK_DIR, "gtsrb.yaml")
with open(yaml_path, 'w') as f:
    f.write(yaml_content)

print(f"✅ Đã tạo file cấu hình: {yaml_path}")
print("\n📄 Nội dung file gtsrb.yaml:")
print("=" * 50)
print(yaml_content[:500])
print("...")

## 📌 Bước 4: Xem thử vài ảnh trong dataset

In [ ]:
import matplotlib.pyplot as plt

# Hiển thị 12 ảnh mẫu từ dataset
fig, axes = plt.subplots(2, 6, figsize=(18, 6))
fig.suptitle("Mẫu ảnh biển báo từ dataset GTSRB", fontsize=16, fontweight='bold')

sample_images = random.sample(train_data, 12)

for idx, (img_path, cls_id, xc, yc, bw, bh, img) in enumerate(sample_images):
    ax = axes[idx // 6][idx % 6]
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    ax.imshow(img_rgb)
    ax.set_title(f"Class {cls_id}\n{GTSRB_CLASSES[cls_id]}", fontsize=8)
    ax.axis('off')

plt.tight_layout()
plt.show()

print(f"\n📊 Thống kê dataset:")
print(f"   Tổng ảnh: {len(all_images):,}")
print(f"   Train: {len(train_data):,} | Val: {len(val_data):,}")
print(f"   Số lớp: {len(GTSRB_CLASSES)}")

## 📌 Bước 5: Huấn luyện YOLOv8 🚀

### Giải thích các tham số:
- **model**: `yolov8n.pt` (nano - nhẹ, nhanh, phù hợp để bắt đầu)
- **epochs**: Số vòng lặp train (50 là đủ cho GTSRB)
- **imgsz**: Kích thước ảnh đầu vào (640 pixels)
- **batch**: Số ảnh xử lý cùng lúc (16 phù hợp GPU T4)

> 💡 **Mẹo:** Sau khi chạy xong, thử đổi model thành `yolov8s.pt` (small) hoặc `yolov8m.pt` (medium) để cải thiện accuracy!

In [ ]:
from ultralytics import YOLO

# ============================================================
# CÁC THAM SỐ CÓ THỂ CHỈNH SỬA
# ============================================================
MODEL_SIZE = "yolov8n.pt"   # Thử: yolov8s.pt, yolov8m.pt, yolov8l.pt
EPOCHS = 50                  # Tăng lên 100 nếu muốn kết quả tốt hơn
IMG_SIZE = 640               # Kích thước ảnh
BATCH_SIZE = 16              # Giảm xuống 8 nếu hết VRAM 
PROJECT_NAME = "bien_bao"    # Tên project
# ============================================================

# Load model pretrained
model = YOLO(MODEL_SIZE)

print(f"🧠 Model: {MODEL_SIZE}")
print(f"📊 Epochs: {EPOCHS}")
print(f"📐 Image size: {IMG_SIZE}")
print(f"📦 Batch size: {BATCH_SIZE}")
print(f"\n🚀 Bắt đầu huấn luyện...\n")

# Huấn luyện
results = model.train(
    data=yaml_path,
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    project=PROJECT_NAME,
    name="train_v1",
    patience=10,           # Dừng sớm nếu không cải thiện sau 10 epochs
    save=True,             # Lưu model
    save_period=10,        # Lưu checkpoint mỗi 10 epochs
    plots=True,            # Tạo biểu đồ
    verbose=True
)

print("\n" + "=" * 50)
print("✅ HUẤN LUYỆN HOÀN TẤT!")
print("=" * 50)

## 📌 Bước 6: Đánh giá kết quả

In [ ]:
from IPython.display import Image, display
import glob

# Tìm thư mục kết quả mới nhất
train_dir = os.path.join(WORK_DIR, PROJECT_NAME, "train_v1")

print("📊 KẾT QUẢ HUẤN LUYỆN")
print("=" * 50)

# Hiển thị biểu đồ training
plots = ["results.png", "confusion_matrix.png", "F1_curve.png", "P_curve.png", "R_curve.png"]

for plot_name in plots:
    plot_path = os.path.join(train_dir, plot_name)
    if os.path.exists(plot_path):
        print(f"\n📈 {plot_name}:")
        display(Image(filename=plot_path, width=800))

In [ ]:
# Đánh giá model trên tập validation
best_model_path = os.path.join(train_dir, "weights", "best.pt")

print(f"📂 Model tốt nhất: {best_model_path}")
print(f"   File size: {os.path.getsize(best_model_path) / 1e6:.1f} MB")

# Chạy validation
best_model = YOLO(best_model_path)
val_results = best_model.val(data=yaml_path, imgsz=IMG_SIZE)

print("\n📊 KẾT QUẢ ĐÁNH GIÁ:")
print("=" * 50)
print(f"   mAP@50:     {val_results.box.map50:.4f}")
print(f"   mAP@50-95:  {val_results.box.map:.4f}")
print(f"   Precision:  {val_results.box.mp:.4f}")
print(f"   Recall:     {val_results.box.mr:.4f}")

## 📌 Bước 7: Test nhận diện trên ảnh mới

In [ ]:
import matplotlib.pyplot as plt

# Lấy vài ảnh từ tập validation để test
val_images = glob.glob(os.path.join(DATASET_DIR, "images", "val", "*.jpg"))[:8]

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
fig.suptitle("🔍 Kết quả nhận diện trên ảnh validation", fontsize=16, fontweight='bold')

for idx, img_path in enumerate(val_images):
    # Chạy inference
    results = best_model.predict(source=img_path, conf=0.5, save=False, verbose=False)
    
    # Vẽ kết quả
    annotated = results[0].plot()
    annotated_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
    
    ax = axes[idx // 4][idx % 4]
    ax.imshow(annotated_rgb)
    
    # Hiển thị tên biển báo
    if results[0].boxes is not None and len(results[0].boxes) > 0:
        cls_id = int(results[0].boxes[0].cls[0])
        conf = float(results[0].boxes[0].conf[0])
        ax.set_title(f"{results[0].names[cls_id]}\n{conf:.1%}", fontsize=9)
    else:
        ax.set_title("Không phát hiện", fontsize=9)
    ax.axis('off')

plt.tight_layout()
plt.show()

## 📌 Bước 8: Upload ảnh riêng để test

In [ ]:
from google.colab import files
import matplotlib.pyplot as plt

print("📤 Chọn ảnh biển báo để test nhận diện:")
uploaded = files.upload()

for filename in uploaded.keys():
    # Chạy nhận diện
    results = best_model.predict(source=filename, conf=0.3, save=False, verbose=False)
    
    # Hiển thị kết quả
    annotated = results[0].plot()
    annotated_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
    
    plt.figure(figsize=(10, 8))
    plt.imshow(annotated_rgb)
    plt.axis('off')
    
    # In kết quả
    if results[0].boxes is not None and len(results[0].boxes) > 0:
        print(f"\n🎯 Kết quả cho {filename}:")
        for box in results[0].boxes:
            cls_id = int(box.cls[0])
            conf = float(box.conf[0])
            print(f"   ✅ {results[0].names[cls_id]} — {conf:.1%}")
    else:
        print(f"   ⚠️ Không phát hiện biển báo trong {filename}")
    
    plt.title(f"Kết quả nhận diện: {filename}", fontsize=14)
    plt.show()

## 📌 Bước 9: Tải model về máy

Sau khi train xong, tải file `best.pt` về để dùng trong ứng dụng demo Streamlit.

In [ ]:
from google.colab import files

# Tải model best.pt về máy
best_pt = os.path.join(train_dir, "weights", "best.pt")
last_pt = os.path.join(train_dir, "weights", "last.pt")

print("📥 Đang tải model về máy...")
print(f"   best.pt: {os.path.getsize(best_pt) / 1e6:.1f} MB")

files.download(best_pt)

print("\n✅ Đã tải best.pt về máy!")
print("\n📋 Bước tiếp theo:")
print("   1. Đặt file best.pt cùng thư mục với Demo_App.py")
print("   2. Chạy: streamlit run Demo_App.py")
print("   3. Mở trình duyệt: http://localhost:8501")
print("\n🎉 Chúc mừng! Bạn đã train xong model nhận diện biển báo!")

---

## 💡 GỢI Ý CẢI TIẾN (Để được điểm cao!)

### 1. Tăng accuracy
- Đổi model lớn hơn: `yolov8s.pt` → `yolov8m.pt` → `yolov8l.pt`
- Tăng epochs: 50 → 100 → 200
- Thêm data augmentation

### 2. Thêm dữ liệu biển báo Việt Nam
- Chụp ảnh biển báo trên đường
- Gán nhãn bằng [Roboflow](https://roboflow.com/) (miễn phí)
- Thêm vào dataset → train lại

### 3. Cải tiến Demo App
- Thêm webcam real-time
- Thêm biểu đồ thống kê
- Làm đẹp giao diện (CSS, animation)

### 4. Tính năng nâng cao
- Tracking biển báo qua video (DeepSORT)
- Cảnh báo âm thanh khi phát hiện biển báo
- Export kết quả ra CSV/PDF

---

**🚦 Cuộc thi Nhận Diện Giao Thông — Ngày hội CNTT BETU 2026**  
**Phụ trách:** Thầy Nguyên Ba Duy